In [4]:
import numpy as np
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [5]:
df = pd.read_csv('qoute_dataset.csv')

In [6]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [7]:
df['quote'][0]

'“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”'

In [8]:
df.shape

(3038, 2)

In [9]:
quotes = df['quote']
quotes.head()

,quote
0,“The world as we have created it is a process ...
1,"“It is our choices, Harry, that show what we t..."
2,“There are only two ways to live your life. On...
3,"“The person, be it gentleman or lady, who has ..."
4,"“Imperfection is beauty, madness is genius and..."


In [10]:
quotes = quotes.str.lower()

In [11]:
import string
translator = str.maketrans('', '', string.punctuation)
quotes = quotes.apply(lambda x: x.translate(translator))

In [12]:
quotes.head()

,quote
0,“the world as we have created it is a process ...
1,“it is our choices harry that show what we tru...
2,“there are only two ways to live your life one...
3,“the person be it gentleman or lady who has no...
4,“imperfection is beauty madness is genius and ...


In [13]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [14]:
from IPython.core.displayhook import tokenize
vocab_size = 10000

tokenizer = Tokenizer(num_words=vocab_size)
tokenizer.fit_on_texts(quotes)

In [15]:
word_index = tokenizer.word_index

print(len(word_index))

8978


In [16]:
sequence = tokenizer.texts_to_sequences(quotes)

In [17]:
quotes[0]

'“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”'

In [18]:
sequence[0]

[713,
 62,
 29,
 19,
 16,
 946,
 10,
 7,
 5,
 1156,
 8,
 70,
 293,
 10,
 145,
 12,
 809,
 104,
 752,
 70,
 2461]

In [19]:
X = []
y = []

for seq in sequence:
  for i in range(1,len(seq)):
    input_seq = seq[:i]
    output_seq = seq[i]
    X.append(input_seq)
    y.append(output_seq)

In [20]:
len(X)
len(y)

85271

In [21]:
max_len = max(len(x) for x in X)
print(max_len)

745


In [22]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [23]:
x_padded = pad_sequences(X, maxlen=max_len, padding='pre')

In [24]:
x_padded

array([[   0,    0,    0, ...,    0,    0,  713],
       [   0,    0,    0, ...,    0,  713,   62],
       [   0,    0,    0, ...,  713,   62,   29],
       ...,
       [   0,    0,    0, ...,    9,   19, 1125],
       [   0,    0,    0, ...,   19, 1125,    3],
       [   0,    0,    0, ..., 1125,    3,  169]], dtype=int32)

In [25]:
y = np.array(y)

In [26]:
x_padded.shape

(85271, 745)

In [27]:
y.shape

(85271,)

In [28]:
from tensorflow.keras.utils import to_categorical

y_cat = to_categorical(y, num_classes = vocab_size)

In [29]:
y_cat.shape

(85271, 10000)

In [30]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,SimpleRNN, LSTM, Dense

In [31]:
embedding_dim = 50
rnn_units = 128

In [32]:
rnn_model = Sequential()

rnn_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedding_dim , input_length = max_len)
)

rnn_model.add(
    SimpleRNN(units=rnn_units)
)

rnn_model.add(
    Dense(units=vocab_size, activation='softmax')
)

rnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [33]:
rnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [34]:
lstm_model = Sequential()
lstm_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedding_dim , input_length = max_len)
)

lstm_model.add(
    LSTM(units=rnn_units)
)
lstm_model.add(
    Dense(units=vocab_size, activation='softmax')
)

lstm_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [35]:
lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [36]:
epochs = 10
batch_size = 128

In [37]:
history_rnn = rnn_model.fit(
    x_padded,
    y_cat,
    epochs=epochs,
    validation_split=0.1,
    batch_size=batch_size
)

Epoch 1/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 51s 76ms/step - accuracy: 0.0452 - loss: 6.7124 - val_accuracy: 0.0620 - val_loss: 6.5701
Epoch 2/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 41s 69ms/step - accuracy: 0.0665 - loss: 6.2580 - val_accuracy: 0.0806 - val_loss: 6.4188
Epoch 3/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 40s 67ms/step - accuracy: 0.0898 - loss: 5.9180 - val_accuracy: 0.0925 - val_loss: 6.3790
Epoch 4/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 40s 67ms/step - accuracy: 0.1042 - loss: 5.6818 - val_accuracy: 0.0952 - val_loss: 6.3905
Epoch 5/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 40s 67ms/step - accuracy: 0.1171 - loss: 5.4726 - val_accuracy: 0.1015 - val_loss: 6.3758
Epoch 6/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 40s 67ms/step - accuracy: 0.1292 - loss: 5.2747 - val_accuracy: 0.1046 - val_loss: 6.4051
Epoch 7/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 40s 67ms/step - accuracy: 0.1396 - loss: 5.0786 - val_accuracy: 0.1093 - val_loss: 6.4338
Epoch 8/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 40s 67ms/step - accuracy: 0.1504 - loss: 4.8887 - 

In [42]:
rnn_model.save("rnn_model.h5")

In [39]:
index_to_word = {}
for word , index in word_index.items():
  index_to_word[index] = word

In [41]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [43]:
def predictor(model, tokenizer , text , max_len):
  text = text.lower()

  seq = tokenizer.texts_to_sequences([text])[0]
  seq = pad_sequences([seq], maxlen=max_len, padding='pre')

  pred = model.predict(seq,verbose = 0)
  pred_index = np.argmax(pred)
  return index_to_word[pred_index]

In [47]:
seed_text = "are you"
next_word = predictor(rnn_model, tokenizer, seed_text, max_len)
print(next_word)

are


In [48]:
def generate_text(model, tokenizer, seed_text, max_len, n_words):
  for _ in range(n_words):
    next_word = predictor(model,tokenizer,seed_text,max_len)
    if next_word == "":
      break
    seed_text += " " + next_word
  return seed_text

In [51]:
seed = "the meaning of life"
generate_text = generate_text(rnn_model, tokenizer, seed, max_len, 10)
print(generate_text)

the meaning of life is a great thing in the world and the most


In [52]:
import pickle

In [53]:
with open('tokenizer.pkl','wb') as f:
  pickle.dump(tokenizer,f)

In [54]:
with open('max_len.pkl','wb') as f:
  pickle.dump(max_len,f)